In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential #neural network
from tensorflow.keras.layers import Dense #hidden layers
from tensorflow.keras.optimizers import SGD #gradient descent

from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score, r2_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 1- Data Exploring

In [ ]:
file_path = '/kaggle/input/eyouth-real-estate-agency-data/RealEstateAgencyData.xlsx'
xls = pd.ExcelFile(file_path)
xls.sheet_names

In [ ]:
df_client = pd.read_excel(xls, sheet_name='Clients')
df_client.head()

In [ ]:
df_client = df_client.dropna(subset=['ClientID'])  # Drop if no Client ID
    
df_client.describe()

In [ ]:
df_agent = pd.read_excel(xls, sheet_name='Agents')
df_agent.head()

In [ ]:
#df_agent['ExperienceLevel'] = pd.cut(df_agent['YearsExperience'], bins=[0,2,5,10,30], labels=['Beginner','Intermediate','Advanced','Expert'])
df_agent.head()

In [ ]:
df_property = pd.read_excel(xls, sheet_name='Properties')
df_property.head()

In [ ]:
df_property.describe()

In [ ]:
df_sales = pd.read_excel(xls, sheet_name='Sales')
df_sales.head()

# 2 - Exploratory Data Analysis

In [ ]:
df_sales['SaleDate'] = pd.to_datetime(df_sales['SaleDate'])
df_sales['SaleMonth'] = df_sales['SaleDate'].dt.month
df_sales['SaleYear'] = df_sales['SaleDate'].dt.year
df_sales['HighValueSale'] = df_sales['SalePrice'] > df_sales['SalePrice'].median()
df_sales.head()

In [ ]:
df_combined = df_sales.merge(df_client, on='ClientID', how='left')\
                      .merge(df_agent, on='AgentID', how='left')\
                      .merge(df_property, on='PropertyID', how='left')
df_combined.head()
df_combined = df_combined.drop('SaleDate', axis=1)

In [ ]:
sns.histplot(df_combined['SalePrice'], kde=True)
plt.title('Distribution of Sale Price')
plt.show()

In [ ]:
# remove objects dtypes with more appropriate form

X_encoded = df_combined.copy()
for col in X_encoded.select_dtypes(include='object').columns:
    X_encoded[col] = X_encoded[col].astype('category')

# picking features and targets

target = X_encoded['SalePrice'].astype(float)

X = X_encoded.drop('SalePrice', axis=1)
y = target

X.head(3)

In [ ]:
X.dtypes

In [ ]:
categories = X.select_dtypes(include='category')
categories.info()

In [ ]:
# drop unnecessary columns
X = X.drop(['FirstName_x','LastName_x','Phone_x','Email_x',
            'FirstName_y','LastName_y','Phone_y','Email_y'], axis=1)

#convert categories into bool features for training
X = pd.get_dummies(X)
X.dtypes

In [ ]:
X.shape

# 3- Label Encoding

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)
y = scaler.fit_transform(y.values.reshape(-1,1))

# 4- Predictive Training

In [ ]:
# train model by 70% 30% then splitting validation into 15% and 15% testing

X_train, X_dummy, y_train, y_dummy = train_test_split(X,
                                                      y,
                                                      shuffle=True,
                                                      train_size=0.7,
                                                      random_state=42)

X_valid, X_test, y_valid, y_test = train_test_split(X_dummy,
                                                    y_dummy,
                                                    shuffle=True,
                                                    train_size=0.5,
                                                    random_state=42)

In [ ]:
# artifical neural network

model = Sequential([
    Dense(8, activation='relu', input_dim = X_train.shape[1]),
    Dense(4, activation='relu'),

    Dense(1, activation='linear')
])

model.compile(optimizer = SGD(learning_rate=0.001), loss='mse')
model.summary()

In [ ]:
# fitting the model in 100 epochs
history = model.fit(X_train, y_train, validation_split = 0.3, epochs = 100, batch_size = 4)

# 5- Evaluation

In [ ]:
# predict models using binary classification

y_pred = model.predict(X_test)

y_test_binary = (y_test > 0.5).astype(int)
y_pred_classes = (y_pred > 0.5).astype(int)

precision = precision_score(y_test_binary, y_pred_classes)
recall = recall_score(y_test_binary, y_pred_classes)
f1 = f1_score(y_test_binary, y_pred_classes)
r2 = r2_score(y_test_binary, y_pred_classes)

print(f"Precision score is: {precision}\nrecall score is: {recall}\nf1 score is: {f1}\nr2: {r2}")

In [ ]:
# ---------- 📊 CONFUSION MATRIX ----------
cm = confusion_matrix(y_test_binary, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# ---------- 📈 ROC CURVE + AUC ----------
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_classes)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')  # Diagonal
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

# 6-Save The Model

In [ ]:
model.save('neural_network_real_estate.h5')